<a href="https://colab.research.google.com/github/duane-edgington/duane-uavs-inference-rf-detr-on-detection-dataset/blob/main/updated_duane_uavs_inference_rf_detr_on_detection_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Train RF-DETR Object Detection on a Custom Dataset

---

[![hf space](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Spaces-blue)](https://huggingface.co/spaces/SkalskiP/RF-DETR)
[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/how-to-finetune-rf-detr-on-detection-dataset.ipynb)
[![roboflow](https://raw.githubusercontent.com/roboflow-ai/notebooks/main/assets/badges/roboflow-blogpost.svg)](https://blog.roboflow.com/rf-detr)
[![code](https://badges.aleen42.com/src/github.svg)](https://github.com/roboflow/rf-detr)

RF-DETR is a real-time, transformer-based object detection model architecture developed by Roboflow and released under the Apache 2.0 license.

![rf-detr-coco-rf100-vl-8](https://media.roboflow.com/rfdetr/pareto.png)

The RF-DETR family of models stands as the quickest and most precise in object detection across all sizes. RF-DETR has achieved over 60 mAP on the Microsoft COCO benchmark, a leading measure of object detection performance. It also sets new records on RF100-VL, a benchmark that shows how well models adapt to real-world problems beyond standard datasets.

The RF-DETR model group includes five sizes: Nano, Small, Medium, Base, and Large. These models offer a range of options for different needs. For example, RF-DETR-Nano is 11 mAP higher than YOLO11-n (on mAP50:95) and runs 0.17 ms faster. Likewise, RF-DETR-Small is 1.8 mAP better than YOLO11-x (the biggest YOLO11 model) and speeds things up by a good 7.77 ms. This wide range of models makes RF-DETR a great choice for many real-world uses, from small devices that need to be super fast to bigger jobs that demand top precision.

RF-DETR is small enough to run on edge devices, making it perfect for deployments that need both high accuracy and real-time performance. You can easily get started with any of the RF-DETR models, as they're ready for training in the cloud with Roboflow or through the free RF-DETR Python package.

## Environment setup

### Configure API Key
#### not needed for inference!

To fine-tune RF-DETR, you need to provide your Roboflow API key. Follow these steps:

- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy` to copy your private API key.
- In Colab, go to the left pane and click on `Secrets` (🔑).
    - Store your Roboflow API Key under the name `ROBOFLOW_API_KEY`.

In [ ]:
import os
from google.colab import userdata
# don't need the API key if using fine tuned RF-DETR model downloaded from huggingface
# os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

### Check GPU availability

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `T4 GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

### Install dependencies

Installs RF-DETR version 1.2.1 or higher (which includes the new Nano, Small, and Medium checkpoints), along with Supervision for benchmarking and Roboflow for pulling datasets and uploading models to the Roboflow platform.

In [ ]:
#!pip install -q rfdetr==1.2.1 supervision==0.27.0 roboflow
!pip install -q rfdetr==1.5.1 supervision==0.27.0 roboflow
#!pip install --upgrade inference

### Download example data

Downloads example images for testing. You can use these or replace them with your own images.

In [ ]:
#!wget -q https://media.roboflow.com/notebooks/examples/dog-2.jpeg
#!wget -q https://media.roboflow.com/notebooks/examples/dog-3.jpeg

import requests
import os

def download_files():
    # URLs to download
    urls = [
        "https://media.roboflow.com/notebooks/examples/dog-2.jpeg",
        "https://media.roboflow.com/notebooks/examples/dog-3.jpeg"
    ]

    for url in urls:
        try:
            # Get the filename from the URL
            filename = os.path.basename(url)

            # Download the file
            response = requests.get(url, stream=True)
            response.raise_for_status()  # Raise an exception for bad status codes

            # Save the file
            with open(filename, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):
                    file.write(chunk)

            print(f"✓ Successfully downloaded {filename}")

        except Exception as e:
            print(f"✗ Error downloading {url}: {e}")

# Execute the download
download_files()

## set threshold for inference

In [ ]:
## set the threshold for inference
THRESHOLD = 0.45

## Inference with Pre-trained COCO Model

Runs inference on an example image using a pretrained RF-DETR Medium model (trained on COCO). To use a different model size, simply replace `RFDETRMedium` with `RFDETRNano`, `RFDETRSmall`, `RFDETRBase` or `RFDETRLarge` as needed.

In [ ]:
import numpy as np
import supervision as sv

from PIL import Image

from rfdetr import RFDETRLarge
from rfdetr.util.coco_classes import COCO_CLASSES

image = Image.open("dog-3.jpeg")

model = RFDETRLarge(resolution=896)
#model = RFDETRLarge(resolution=704)
model.optimize_for_inference() # Removed to resolve tracing error

detections = model.predict(image, threshold=THRESHOLD)


color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])
text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)

bbox_annotator = sv.BoxAnnotator(color=color, thickness=thickness)
label_annotator = sv.LabelAnnotator(
    color=color,
    text_color=sv.Color.BLACK,
    text_scale=text_scale,
    smart_position=True
)

labels = [
    f"{COCO_CLASSES[class_id]} {confidence:.2f}"
    for class_id, confidence
    in zip(detections.class_id, detections.confidence)
]

annotated_image = image.copy()
annotated_image = bbox_annotator.annotate(annotated_image, detections)
annotated_image = label_annotator.annotate(annotated_image, detections, labels)
annotated_image.thumbnail((800, 800))
annotated_image

## Download Dataset from Roboflow Universe

RF-DETR expects the dataset to be in COCO format. Divide your dataset into three subdirectories: `train`, `valid`, and `test`. Each subdirectory should contain its own `_annotations.coco.json` file that holds the annotations for that particular split, along with the corresponding image files. Below is an example of the directory structure:

```
dataset/
├── train/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
├── valid/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
└── test/
    ├── _annotations.coco.json
    ├── image1.jpg
    ├── image2.jpg
    └── ... (other image files)
```

[Roboflow](https://roboflow.com/annotate) allows you to create object detection datasets from scratch or convert existing datasets from formats like YOLO, and then export them in COCO JSON format for training. You can also explore [Roboflow Universe](https://universe.roboflow.com/) to find pre-labeled datasets for a range of use cases.

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
# Slice size for supervision InferenceSlicer (default: 1024 pixels)
SLICE_SIZE = 896

In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True) # This will prompt for authorization.

# This will create the uavs files if they don't exist.
folders =  ["uavs2/"]
for folder in folders:
  path = "/content/drive/MyDrive/" + folder
  if not os.path.exists(path): # Create the folder if it does not exist
    os.mkdir(path)

In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets

from google.colab import userdata

uavs_folder = "/content/drive/MyDrive/uavs2/"

!mkdir /content/datasets/savedir/
!mkdir /content/datasets/savedir/images/


In [ ]:
## rf-detr expects
!mkdir /content/datasets/test/
!mkdir /content/datasets/test/images/

#get the images directory
!rsync -a "/content/drive/MyDrive/uavs2/images/" "/content/datasets/savedir/images/"

#get the data.yaml file
!cp "/content/drive/MyDrive/uavs2/data.yaml" "/content/datasets/data.yaml"

#move the data to the expected directories
!rsync -a "/content/datasets/savedir/images/" "/content/datasets/test/images/"

!ls /content/datasets/

In [ ]:
# this is used only if input is in yolo format. Not needed if conversion to coco done externally
import os
import json
from PIL import Image

def convert_yolo_to_coco(yolo_images_path, yolo_labels_path, output_coco_path, class_names):
    coco_data = {
        "info": {
            "year": "2025",
            "version": "0",
            "description": "UAVS",
            "contributor": "",
            "url": "https://public.roboflow.com/object-detection/undefined",
            "date_created": "2025-07-24T06:49:14+00:00"
        },
        "licenses": [
            {
                "id": 1,
                "url": "https://creativecommons.org/licenses/by/4.0/",
                "name": "CC BY 4.0"
            }
        ],
        "images": [],
        "annotations": [],
        "categories": []
    }

    # Populate categories
    for i, name in enumerate(class_names):
        coco_data["categories"].append({"id": i, "name": name, "supercategory": "none"})

    image_id = 0
    annotation_id = 0

    for img_filename in os.listdir(yolo_images_path):
        if not img_filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        img_path = os.path.join(yolo_images_path, img_filename)
        label_filename = os.path.splitext(img_filename)[0] + '.txt'
        label_path = os.path.join(yolo_labels_path, label_filename)

        with Image.open(img_path) as img:
            width, height = img.size

        coco_data["images"].append({
            "id": image_id,
            "file_name": img_filename,
            "width": width,
            "height": height
        })

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    class_id, x_center, y_center, bbox_width, bbox_height = map(float, parts)

                    # Convert normalized YOLO to absolute COCO
                    abs_x_center = x_center * width
                    abs_y_center = y_center * height
                    abs_bbox_width = bbox_width * width
                    abs_bbox_height = bbox_height * height

                    x_min = abs_x_center - (abs_bbox_width / 2)
                    y_min = abs_y_center - (abs_bbox_height / 2)

                    coco_data["annotations"].append({
                        "id": annotation_id,
                        "image_id": image_id,
                        "category_id": int(class_id),
                        "bbox": [x_min, y_min, abs_bbox_width, abs_bbox_height],
                        "area": abs_bbox_width * abs_bbox_height,
                        "iscrowd": 0
                    })
                    annotation_id += 1
        image_id += 1

    with open(output_coco_path, 'w') as f:
        json.dump(coco_data, f, indent=4)



In [ ]:
# conversion to coco done externally. Just copy them into the colab instance
!cp "/content/drive/MyDrive/uavs2/_annotations.coco.json" "/content/datasets/test/_annotations.coco.json"

In [ ]:
# First, clean up gpu memory to make sure we have as much space as possible

import gc
import torch
import weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):

    if not torch.cuda.is_available():
        if verbose:
            print("[INFO] CUDA is not available. No GPU cleanup needed.")
        return

    def get_memory_stats():
        allocated = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        return allocated, reserved

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

    # Ensure we drop all strong references
    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("[WARNING] Object not fully garbage collected yet.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

In [ ]:
cleanup_gpu_memory(model, verbose=True)

## Deploy a Trained RF-DETR Model

Deploying to Roboflow allows you to create multi-step computer vision applications that run both in the cloud and your own hardware. Please wait a moment while Roboflow indexes your model.

## Evaluate Fine-tuned RF-DETR Model

Before benchmarking the model, we need to load the best saved checkpoint. To ensure it fits on the GPU, we first need to free up GPU memory. This involves deleting any remaining references to previously used objects, triggering Python’s garbage collector, and clearing the CUDA memory cache.

We load the best-performing model from the `checkpoint_best_total.pth` file using the `RFDETRMedium` class. This checkpoint contains the trained weights from our most successful training run. After loading, we call `optimize_for_inference()`, which prepares the model for efficient inference.

In [ ]:
dataset = "/content/datasets/"
from rfdetr import RFDETRLarge
model = RFDETRLarge(pretrain_weights="/content/drive/MyDrive/uavs2/checkpoint_best_total.pth",resolution=896, num_classes=1)
model.optimize_for_inference()

In [ ]:
import supervision as sv

ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"{dataset}/test/images",
    annotations_path=f"{dataset}/test/_annotations.coco.json",
)

In [ ]:
import supervision as sv
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision

# Set up InferenceSlicer for tiled inference on large images.
# SLICE_SIZE controls the width and height of each tile (default: 1024 pixels).
def slicer_callback(tile):
    #return model.predict(tile, threshold=0)
    return model.predict(tile, threshold=THRESHOLD)

slicer = sv.InferenceSlicer(
    callback=slicer_callback,
    slice_wh=SLICE_SIZE,
    overlap_wh=SLICE_SIZE // 4,  # 25% overlap between tiles
)

targets = []
predictions = []

for path, image, annotations in tqdm(ds):
    image = Image.open(path)
    #detections = model.predict(image, threshold=0)
    detections = slicer(image)  # Use InferenceSlicer instead of model.predict directly


    targets.append(annotations)
    predictions.append(detections)

In [ ]:
map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()
print(map_result)

In [ ]:
print(type(targets[0]))
print(type(predictions[0]))

In [ ]:
# make the confusion matrix and display it
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import supervision as sv
import torch
import torchvision
import os

# Define a function to match predictions to ground truth
def match_detections(ground_truth, predictions, iou_threshold=0.5):
    if len(ground_truth) == 0 and len(predictions) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=int)
    elif len(ground_truth) == 0:
        # No ground truth, all predictions are false positives
        # Return empty array for matched_gt and matched_pred, and all prediction indices for fp
        return np.array([], dtype=int), np.array([], dtype=int), np.arange(len(predictions), dtype=int)
    elif len(predictions) == 0:
        # No predictions, all ground truth are false negatives
        # Return all ground truth indices for matched_gt, and empty arrays for matched_pred and fp
        return np.arange(len(ground_truth), dtype=int), np.array([], dtype=int), np.array([], dtype=int)

    # Convert bounding boxes to torch tensors
    ground_truth_boxes = torch.from_numpy(ground_truth.xyxy)
    predictions_boxes = torch.from_numpy(predictions.xyxy)

    # Calculate IoU matrix using torchvision
    # box_iou_batch returns a [N, M] matrix where N is ground truth and M is predictions
    iou_matrix = torchvision.ops.box_iou(ground_truth_boxes, predictions_boxes)

    # Find the best match for each ground truth detection
    # We need to find the maximum IoU for each ground truth box
    # And ensure that each prediction box is matched at most once

    # Get the index of the best prediction for each ground truth box
    best_iou_for_gt, best_pred_indices_for_gt = iou_matrix.max(dim=1)

    matched_ground_truth_indices = []
    matched_prediction_indices = []
    prediction_matched = [False] * len(predictions)

    # Iterate through ground truth boxes and their best matching predictions
    for i in range(len(ground_truth)):
        best_iou = best_iou_for_gt[i].item()
        best_pred_index = best_pred_indices_for_gt[i].item()

        # Check if the best match is above the threshold and the prediction hasn't been matched yet
        if best_iou >= iou_threshold and not prediction_matched[best_pred_index]:
            matched_ground_truth_indices.append(i)
            matched_prediction_indices.append(best_pred_index)
            prediction_matched[best_pred_index] = True

    # Identify false positives (predictions not matched to any ground truth)
    false_positive_indices = [j for j in range(len(predictions)) if not prediction_matched[j]]

    # Return as numpy arrays
    return np.array(matched_ground_truth_indices, dtype=int), np.array(matched_prediction_indices, dtype=int), np.array(false_positive_indices, dtype=int)

# Prepare data for confusion matrix
true_labels = []
predicted_labels = []

# Iterate through the dataset
for gt_detections, pred_detections in zip(targets, predictions):
    # Ensure that gt_detections and pred_detections are not None before processing
    if gt_detections is None or pred_detections is None:
        continue

    # Filter out predictions below a certain confidence threshold if desired
    # For confusion matrix calculation, it's often better to include all predictions
    # and let the IOU threshold handle the matching.
    # If you want to filter by confidence, you can do it here:
    # pred_detections = pred_detections[pred_detections.confidence > confidence_threshold]


    matched_gt_indices, matched_pred_indices, false_positive_indices = match_detections(
        gt_detections, pred_detections, iou_threshold=0.5 # Adjust IoU threshold as needed
    )

    # Add true positives
    for gt_idx, pred_idx in zip(matched_gt_indices, matched_pred_indices):
        true_labels.append(gt_detections.class_id[gt_idx])
        predicted_labels.append(pred_detections.class_id[pred_idx])

    # Add false positives (predicted class, but no true class)
    for fp_idx in false_positive_indices:
        true_labels.append(-1)  # Use -1 to represent 'no object' in ground truth
        predicted_labels.append(pred_detections.class_id[fp_idx])

    # Add false negatives (true class, but no predicted class)
    # Iterate through ground truth detections that were not matched
    all_gt_indices = set(range(len(gt_detections)))
    false_negative_indices = list(all_gt_indices - set(matched_gt_indices))
    for fn_idx in false_negative_indices:
        true_labels.append(gt_detections.class_id[fn_idx])
        predicted_labels.append(-1) # Use -1 to represent 'no object' in prediction


# Include -1 in class names for the confusion matrix to represent 'no object'
# Adjust class_names to include a label for the background or 'no object' class if needed
# Assuming class_names is a list of strings where the index corresponds to the class_id
# If your class_ids are 0-indexed, and you have 'n' classes, the labels will be 0 to n-1.
# We add -1 for 'no object', so the unique labels will be -1 and 0 to n-1.
# Let's assume your class_ids are 0-indexed and you have 1 class ("object").
# So class_ids are 0. The unique labels will be -1 and 0.
# The confusion matrix will have dimensions (num_classes + 1) x (num_classes + 1).

# Determine the unique labels present in the data
unique_labels = sorted(list(set(true_labels + predicted_labels)))

# Map original class_ids to indices for the confusion matrix, including -1
label_to_index = {label: i for i, label in enumerate(unique_labels)}

# Convert labels to indices
true_indices = [label_to_index[label] for label in true_labels]
predicted_indices = [label_to_index[label] for label in predicted_labels]

# Calculate the confusion matrix
# Ensure that the labels parameter in confusion_matrix includes all unique labels
cm = confusion_matrix(true_indices, predicted_indices, labels=list(label_to_index.values()))

# Define labels for the confusion matrix plot, including 'No Object'
# Ensure that the 'No Object' label is correctly placed based on the unique_labels
cm_labels = [None] * len(unique_labels)

# Define class_names using ds.classes
class_names = ds.classes

for label, index in label_to_index.items():
    if label == -1:
        cm_labels[index] = 'No Object'
    else:
        # Assuming class_names is indexed by original class_id (0 in this case for "object")
        # Need to handle the mapping from original class_id to the index in class_names if class_names is not [object]
        # Since class_names = ["object"], class_id 0 maps to index 0.
        # If there were more classes, this mapping would need to be more robust.
        try:
             cm_labels[index] = class_names[label]
        except IndexError:
             # Handle cases where a class_id exists in detections but not in class_names
             cm_labels[index] = f"Unknown Class {label}"


# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

# Show all ticks and label them with the respective list entries
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=cm_labels, yticklabels=cm_labels,
       title='Confusion Matrix',
       ylabel='True label',
       xlabel='Predicted label')

# Rotate the tick labels and set their alignment.
plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
         rotation_mode="anchor")

# Loop over data dimensions and create text annotations.
fmt = 'd'
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], fmt),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")

fig.tight_layout()

# Define the output path
output_path = '/content/datasets/output/confusion_matrix.png'

# Ensure the output directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the plot
plt.savefig(output_path)

# Show the plot

plt.show()

In [ ]:
import shutil

#save the confusion matrix
source = "/content/datasets/output/confusion_matrix.png"
destination = "/content/drive/MyDrive/uavs2/output/sliced_threshold_45_confusion_matrix.png"

try:
    # Ensure destination directory exists
    os.makedirs(os.path.dirname(destination), exist_ok=True)

    # Copy the file
    shutil.copy2(source, destination)
    print(f"Successfully copied {source} to {destination}")

except FileNotFoundError:
    print(f"Error: Source file {source} not found")
except Exception as e:
    print(f"Error copying file: {e}")

## Run Inference with Fine-tuned RF-DETR Model

In [ ]:
from rfdetr import RFDETRLarge
import supervision as sv
from PIL import Image

path, image, annotations = ds[51] #17
image = Image.open(path)

def slicer_rf1_callback(tile):
    return model.predict(tile, threshold=THRESHOLD)

slicer_rf1 = sv.InferenceSlicer(
    callback=slicer_rf1_callback,
    slice_wh=SLICE_SIZE,
    overlap_wh=SLICE_SIZE // 4,  # 25% overlap between tiles
)

detections = slicer_rf1(image)  # Use InferenceSlicer instead of model_rf.predict directly

#detections = model.predict(image, threshold=THRESHOLD)

text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

bbox_annotator = sv.BoxAnnotator(color=color,thickness=thickness)
label_annotator = sv.LabelAnnotator(
    color=color,
    text_color=sv.Color.BLACK,
    text_scale=text_scale)

annotations_labels = [
    f"{ds.classes[class_id]}"
    for class_id
    in annotations.class_id
]

detections_labels = [
    f"{ds.classes[class_id]} {confidence:.2f}"
    for class_id, confidence
    in zip(detections.class_id, detections.confidence)
]

annotation_image = image.copy()
annotation_image = bbox_annotator.annotate(annotation_image, annotations)
annotation_image = label_annotator.annotate(annotation_image, annotations, annotations_labels)

detections_image = image.copy()
detections_image = bbox_annotator.annotate(detections_image, detections)
detections_image = label_annotator.annotate(detections_image, detections, detections_labels)

sv.plot_images_grid(images=[annotation_image, detections_image], grid_size=(1, 2), titles=["Annotation", "Detection"])

In [ ]:
#!pip install -q inference

#!pip install -q inference-gpu==0.58.1
#!pip install -q inference-gpu==0.64.5
!pip install -q inference-gpu==1.0.2

In [ ]:
from inference import get_model

#MODEL_ID = "basketball-player-detection-2/13"
#model_rf = get_model(model_id=MODEL_ID, api_key = userdata.get("ROBOFLOW_API_KEY"))

model_rf = RFDETRLarge(pretrain_weights="/content/drive/MyDrive/uavs2/checkpoint_best_total.pth",resolution=896, num_classes=1)
model_rf.optimize_for_inference()

In [ ]:
import supervision as sv
from PIL import Image

path, image, annotations = ds[51]
image = Image.open(path)

# Set up InferenceSlicer for tiled inference using the fine-tuned model.
# SLICE_SIZE controls the width and height of each tile (default: 1024 pixels).
def slicer_rf_callback(tile):
    return model_rf.predict(tile, threshold=THRESHOLD)

slicer_rf = sv.InferenceSlicer(
    callback=slicer_rf_callback,
    slice_wh=SLICE_SIZE,
    overlap_wh=SLICE_SIZE // 4,  # 25% overlap between tiles
)

detections = slicer_rf(image)  # Use InferenceSlicer instead of model_rf.predict directly

#detections = model_rf.predict(image, threshold=THRESHOLD)

text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

bbox_annotator = sv.BoxAnnotator(color=color,thickness=thickness)
label_annotator = sv.LabelAnnotator(
    color=color,
    text_color=sv.Color.BLACK,
    text_scale=text_scale)

annotations_labels = [
    f"{ds.classes[class_id]}"
    for class_id
    in annotations.class_id
]

annotation_image = image.copy()
annotation_image = bbox_annotator.annotate(annotation_image, annotations)
annotation_image = label_annotator.annotate(annotation_image, annotations, annotations_labels)

detections_image = image.copy()
detections_image = bbox_annotator.annotate(detections_image, detections)
detections_image = label_annotator.annotate(detections_image, detections)

sv.plot_images_grid(images=[annotation_image, detections_image], grid_size=(1, 2), titles=["Annotation", "Detection"])

In [ ]:
## Here's a notebook cell that runs inference over the entire test dataset and
## saves both the annotated images and COCO-format detection results:


## Run Inference on Entire Test Dataset and Save Results in COCO Format

import os
import json
from PIL import Image
from tqdm import tqdm
import supervision as sv
from rfdetr import RFDETRLarge
import shutil
from datetime import datetime

# Define paths
TEST_IMAGES_DIR = "/content/datasets/test/images"
OUTPUT_DIR = "/content/datasets/output/inference"
ANNOTATED_IMAGES_DIR = os.path.join(OUTPUT_DIR, "annotated_images")
COCO_ANNOTATIONS_PATH = os.path.join(OUTPUT_DIR, "predictions_coco.json")

# Create output directories
os.makedirs(ANNOTATED_IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the fine-tuned model (same as before)
model = RFDETRLarge(
    pretrain_weights="/content/drive/MyDrive/uavs2/checkpoint_best_total.pth",
    resolution=896,
    num_classes=1
)
model.optimize_for_inference()

# Set up the slicer for high-resolution images
SLICE_SIZE = 896

def slicer_callback(tile):
    return model.predict(tile, threshold=THRESHOLD)

slicer = sv.InferenceSlicer(
    callback=slicer_callback,
    slice_wh=SLICE_SIZE,
    overlap_wh=SLICE_SIZE // 4,  # 25% overlap between tiles
)

# Get all test images
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
test_images = [
    f for f in os.listdir(TEST_IMAGES_DIR)
    if f.lower().endswith(image_extensions)
]
test_images.sort()  # Sort for consistent ordering, Note: this is different from ordering above.

print(f"Found {len(test_images)} images in test set")
print(f"Output will be saved to: {OUTPUT_DIR}")

# Initialize COCO format data structure
coco_data = {
    "info": {
        "description": "RF-DETR Inference Results",
        "date_created": datetime.now().isoformat(),
        "model": "RFDETRLarge",
        "threshold": THRESHOLD
    },
    "images": [],
    "annotations": [],
    "categories": [
        {
            "id": 0,
            "name": "object",
            "supercategory": "none"
        }
    ]
}

# Keep track of annotation IDs
annotation_id = 0
image_id = 0

# Process each image
for img_filename in tqdm(test_images, desc="Processing images"):
    img_path = os.path.join(TEST_IMAGES_DIR, img_filename)

    # Open image
    image = Image.open(img_path).convert("RGB")
    width, height = image.size

    # Run inference with slicer
    detections = slicer(image)

    # Add image info to COCO data
    coco_data["images"].append({
        "id": image_id,
        "file_name": img_filename,
        "width": width,
        "height": height
    })

    # Add annotations to COCO data
    if len(detections) > 0:
        # Convert detections to COCO format
        # detections.xyxy gives [x1, y1, x2, y2] format
        xyxy = detections.xyxy
        confidence = detections.confidence
        class_ids = detections.class_id

        for i in range(len(detections)):
            x1, y1, x2, y2 = xyxy[i]
            bbox_width = x2 - x1
            bbox_height = y2 - y1
            area = bbox_width * bbox_height

            coco_data["annotations"].append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": int(class_ids[i]) if class_ids is not None else 0,
                "bbox": [float(x1), float(y1), float(bbox_width), float(bbox_height)],
                "area": float(area),
                "score": float(confidence[i]) if confidence is not None else 1.0,
                "iscrowd": 0
            })
            annotation_id += 1

    # Create and save annotated image
    # Prepare visualization settings
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=(width, height))
    thickness = sv.calculate_optimal_line_thickness(resolution_wh=(width, height))
    color = (
        sv.ColorPalette.default()
        if hasattr(sv.ColorPalette, 'default')
        else sv.ColorPalette.from_hex([
            "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
            "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
        ])
    )

    bbox_annotator = sv.BoxAnnotator(color=color, thickness=thickness)
    label_annotator = sv.LabelAnnotator(
        color=color,
        text_color=sv.Color.BLACK,
        text_scale=text_scale,
        smart_position=True
    )

    # Create labels for detections
    if len(detections) > 0:
        labels = [
            f"object {conf:.2f}" if conf else "object"
            for conf in (detections.confidence if detections.confidence is not None else [None] * len(detections))
        ]

        # Annotate image
        annotated_image = image.copy()
        annotated_image = bbox_annotator.annotate(annotated_image, detections)
        annotated_image = label_annotator.annotate(annotated_image, detections, labels)
    else:
        annotated_image = image.copy()

    # Save annotated image
    output_img_path = os.path.join(ANNOTATED_IMAGES_DIR, img_filename)
    annotated_image.save(output_img_path)

    # Increment image ID
    image_id += 1

# Save COCO annotations JSON file
with open(COCO_ANNOTATIONS_PATH, 'w') as f:
    json.dump(coco_data, f, indent=2)

print(f"\n✅ Inference complete!")
print(f"   - Processed {len(test_images)} images")
print(f"   - Found {annotation_id} total detections")
print(f"   - Annotated images saved to: {ANNOTATED_IMAGES_DIR}")
print(f"   - COCO predictions saved to: {COCO_ANNOTATIONS_PATH}")

# Also save a summary text file
summary_path = os.path.join(OUTPUT_DIR, "inference_summary.txt")
with open(summary_path, 'w') as f:
    f.write(f"Inference Summary\n")
    f.write(f"================\n")
    f.write(f"Date: {datetime.now().isoformat()}\n")
    f.write(f"Model: RFDETRLarge\n")
    f.write(f"Confidence Threshold: {THRESHOLD}\n")
    f.write(f"Slice Size: {SLICE_SIZE}\n")
    f.write(f"Total Images Processed: {len(test_images)}\n")
    f.write(f"Total Detections: {annotation_id}\n")
    f.write(f"Output Directory: {OUTPUT_DIR}\n")

print(f"   - Summary saved to: {summary_path}")

# Optional: Copy results to Google Drive
try:
    DRIVE_OUTPUT = "/content/drive/MyDrive/uavs2/inference_results"
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

    # Copy results to Drive
    if os.path.exists(OUTPUT_DIR):
        shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
        print(f"   - Results also copied to Google Drive: {DRIVE_OUTPUT}")
except Exception as e:
    print(f"   - Note: Could not copy to Google Drive: {e}")

# Display sample of an image with detections
if len(test_images) > 0 and annotation_id > 0:
    from IPython.display import display
    sample_img_path = os.path.join(ANNOTATED_IMAGES_DIR, test_images[0]) # selected image 0 to display
    if os.path.exists(sample_img_path):
        display(Image.open(sample_img_path))
        print(f"\n📸 Preview: {test_images[0]} (showing annotated image)")


## This cell will:

## 1. **Load the fine-tuned RF-DETR model** (same as the single-image inference cell)
## 2. **Process all images** in `/content/datasets/test/images`
## 3. **Save annotated images** to `/content/datasets/output/inference/annotated_images/`
## 4. **Save COCO-format detections** as a JSON file at `/content/datasets/output/inference/predictions_coco.json`
## 5. **Include bounding box coordinates** in standard COCO format: `[x_min, y_min, width, height]`
## 6. **Save a summary text file** with metadata about the inference run
## 7. **Optionally copy results to Google Drive** for permanent storage

## The COCO JSON file structure follows the standard COCO format, containing:
## - `images`: List of images with IDs, filenames, and dimensions
## - `annotations`: List of detections with image_id, bbox coordinates, area, and confidence scores
## - `categories`: Class definitions (your "object" class)

## You can then use this COCO file with standard COCO evaluation tools or load it with supervision library for further analysis.



In [ ]:
if os.path.exists(sample_img_path):
        display(Image.open(sample_img_path))
        print(f"\n📸 Preview: {test_images[0]} (showing annotated image)")


In [ ]:
'''# save results

# Connect Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True) # This will prompt for authorization.

# This will create the rfdetr files if they don't exist.
folders =  ["rfdetr/"]
for folder in folders:
  path = "/content/drive/MyDrive/" + folder
  if not os.path.exists(path): # Create the folder if it does not exist
    os.mkdir(path)

'''

In [ ]:
'''# save results

!mkdir "/content/drive/MyDrive/uavs/datasets/"
!mkdir "/content/drive/MyDrive/uavs/datasets/output/"
!cp -r "/content/datasets/output/" "/content/drive/MyDrive/uavs/datasets/output/"

'''

<div align="center">
  <p>
    Looking for more tutorials or have questions?
    Check out our <a href="https://github.com/roboflow/notebooks">GitHub repo</a> for more notebooks,
    or visit our <a href="https://discord.gg/GbfgXGJ8Bk">discord</a>.
  </p>
  
  <p>
    <strong>If you found this helpful, please consider giving us a ⭐
    <a href="https://github.com/roboflow/notebooks">on GitHub</a>!</strong>
  </p>

</div>